# Retirement Readiness — Data Preprocessing

Notebook 1 explored the dataset and identified what needs to change before a model can be
trained. This notebook carries out those changes and produces a clean, numeric feature matrix.

Raw customer records cannot be used directly: some values are missing, some columns are text
rather than numbers, some must not be used at all, and the numeric columns sit on very different
scales. Each issue is handled below, and every step is packaged into one object so the same
transformation can be applied to new customers later.

One rule shapes the notebook: any step that *learns* something from the data — a median, a
category list — learns it from the training customers only. Otherwise the test set would
influence the model and the final accuracy estimate would be too optimistic.

### Decisions carried forward from the EDA

| Finding | Action |
|---|---|
| `Funding_Gap`, `Readiness_Score` and `RetirementReady` are calculated from the target | Removed from the feature set |
| `CustomerID` identifies a customer but does not describe one | Removed |
| 150 records are exact duplicates | Removed before the split |
| Twelve numeric columns have roughly 1.5% missing values | Median imputation fitted on training data |
| The target is strongly right-skewed | Natural log transform |
| `YearsUntilRetirement` is not provided directly | Engineered from `DesiredRetirementAge - Age` |
| `Age` and `YearsExperience` correlate at +0.94 | `CareerStartAge` engineered |
| The education gradient is not monotonic in the target | One-hot encoding |

---

# 2. Load Data

The project uses the following layout, with the notebook in `notebooks/` and the reusable
feature-engineering code in `src/` at the repository root:

```
Retirement-Readiness-Predictor/
├── data/retirement_dataset_v2.csv
├── notebooks/02_data_preprocessing.ipynb
├── src/feature_engineering.py
├── artifacts/
└── figures/
```

The cells below locate the repository root rather than relying on a fixed path, so the notebook
runs whether it is started from the root or from `notebooks/`. On Colab, clone the repository
first and the same lookup finds it.

In [ ]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn import set_config
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
import joblib

# Display fitted pipelines as a diagram rather than a block of text.
set_config(display="diagram")

print("pandas       ", pd.__version__)
print("numpy        ", np.__version__)
print("scikit-learn ", sklearn.__version__)

In [ ]:
# The repository root is the folder that contains src/. Checking a short list
# of candidates keeps the notebook working from either the root or notebooks/.
CANDIDATE_ROOTS = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/Retirement-Readiness-Predictor"),   # Colab, after cloning
]

PROJECT_ROOT = None
for candidate in CANDIDATE_ROOTS:
    if (candidate / "src" / "feature_engineering.py").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root (the folder containing src/). Checked: "
        + ", ".join(str(path) for path in CANDIDATE_ROOTS)
    )

# Make src/ importable no matter where the notebook was launched from.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
FIGURES_DIR = PROJECT_ROOT / "figures"
ARTIFACTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
# The feature-engineering code lives in src/ so that the saved pipeline can be
# reloaded later without needing this notebook.
from src.feature_engineering import (
    ENGINEERED_FEATURES,
    SOURCE_COLUMNS_TO_DROP,
    add_engineered_features,
    engineered_feature_names,
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300})

NAVY = "#1F3B73"
TEAL = "#2A9D8F"
AMBER = "#E9A13B"
CORAL = "#E76F51"
SLATE = "#6C757D"

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "retirement_dataset_v2.csv"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "retirement_dataset_v2.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"retirement_dataset_v2.csv not found under {PROJECT_ROOT}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

---

# 3. Separate Features and Target

The target is `Expected_Retirement_Fund`, the projected value of a customer's retirement pot. It
is held in its own object, separate from the predictors, so that none of the preprocessing steps
below can ever be applied to it by accident.

The EDA showed the target is strongly right-skewed: most customers cluster at lower values while
a small number reach into the tens of millions. Model training will therefore use the natural
logarithm of the target, which compresses that long tail. The original dollar values are kept
alongside it so predictions can be converted back to business units in Notebook 3.

In [ ]:
TARGET = "Expected_Retirement_Fund"

# A missing or non-positive target could not be logged or used for training.
print(f"Missing target values:      {df[TARGET].isna().sum()}")
print(f"Non-positive target values: {(df[TARGET] <= 0).sum()}")
print(f"Skewness in dollars:        {df[TARGET].skew():+.4f}")
print(f"Skewness after log:         {np.log(df[TARGET]).skew():+.4f}")

---

# 4. Remove IDs and Leakage

### What

Drop `CustomerID`, `Funding_Gap`, `Readiness_Score` and `RetirementReady` from the predictors.

### Why

`CustomerID` identifies the customer but says nothing about their financial situation, and a
flexible model can still memorise it.

The other three are business figures calculated *from* the target once a projection exists:
`Funding_Gap` is the projected fund minus the goal, `Readiness_Score` is the fund divided by the
goal, and `RetirementReady` flags whether the gap is positive. Giving a model any of them hands it
the answer, and none of them exists when a real prediction is needed. The cell below reconfirms
the two arithmetic identities.

### Decision

These four leave the feature set. The business figures are still produced, but recalculated from
the model's prediction afterwards.

### `Retirement_Fund_Goal` is kept

A customer's goal is set with their adviser before any projection is produced, so the project
assumes it is known at prediction time and keeps it as an input. Notebook 3 measures how much it
matters by training one variant without it.

In [ ]:
IDENTIFIER_COLUMNS = ["CustomerID"]
KPI_COLUMNS = ["Readiness_Score", "Funding_Gap", "RetirementReady"]

# Kept as an input, recorded here so Notebook 3 can run the ablation.
ABLATION_REVIEW_COLUMNS = ["Retirement_Fund_Goal"]

gap_error = (df[TARGET] - df.Retirement_Fund_Goal - df.Funding_Gap).abs().max()
ready_match = ((df.Funding_Gap >= 0).astype(int) == df.RetirementReady).mean()

print(f"max |Funding_Gap - (target - goal)|:    {gap_error:.10f}")
print(f"RetirementReady == 1[Funding_Gap >= 0]: {ready_match:.4%} of rows")

In [ ]:
columns_to_exclude = IDENTIFIER_COLUMNS + KPI_COLUMNS + [TARGET]

X = df.drop(columns=columns_to_exclude)
y = df[TARGET]

# Column roles come from the dtypes, so a change to the source file cannot
# leave a column silently unprocessed.
CATEGORICAL_FEATURES = X.select_dtypes(include="object").columns.tolist()
NUMERIC_FEATURES = X.select_dtypes(include=np.number).columns.tolist()

print(f"Predictor columns: {X.shape[1]}  "
      f"({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")
print("Excluded:", ", ".join(columns_to_exclude))

---

# 5. Remove Duplicates

The file contains 150 records that are exact copies of other records. Duplicate observations can
give the same customer extra weight during training and can also cause the same customer to
appear in both the training and test data, which makes the test score look better than it is. We
therefore remove exact duplicates before splitting.

Duplicates are identified on the complete original record, and the same rows are removed from
both `X` and `y` so the two stay aligned.

In [ ]:
duplicate_mask = df.duplicated(keep="first")
rows_to_keep = ~duplicate_mask

print(f"Exact duplicate records: {duplicate_mask.sum()}")
print(f"Rows before: {len(X):,}")

X = X[rows_to_keep]
y = y[rows_to_keep]

print(f"Rows after:  {len(X):,}")
print(f"Duplicates remaining in X: {X.duplicated().sum()}")
print(f"X and y aligned: {X.index.equals(y.index)}")

---

# 6. Train/Test Split

Twenty percent of customers are set aside and not used again until the model is evaluated. The
split is performed before imputation, scaling or encoding so that information from the test set
cannot influence the preprocessing parameters learned from the training data.

`random_state=42` fixes the selection, so re-running the notebook produces the same split and
model comparisons stay reproducible. The target is a continuous amount rather than a category, so
there is no class label to stratify on; the split is checked for balance instead.

In [ ]:
X_train, X_test, y_train_dollars, y_test_dollars = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

# The target must be present and strictly positive because we model its natural logarithm.
assert y_train_dollars.notna().all(), "Training target contains missing values."
assert y_test_dollars.notna().all(), "Test target contains missing values."
assert (y_train_dollars > 0).all(), "Training target must be strictly positive."
assert (y_test_dollars > 0).all(), "Test target must be strictly positive."

# Model training uses the log target; dollars are kept for later reporting.
y_train = np.log(y_train_dollars)
y_test = np.log(y_test_dollars)

print(f"Training set: {X_train.shape[0]:,} rows ({X_train.shape[0] / len(X):.0%})")
print(f"Test set:     {X_test.shape[0]:,} rows ({X_test.shape[0] / len(X):.0%})")
print(f"y_train (log): mean {y_train.mean():.4f}, std {y_train.std():.4f}")
print(f"y_test  (log): mean {y_test.mean():.4f}, std {y_test.std():.4f}")

In [ ]:
# A quick balance check: compare the middle value of several features across
# the two samples. Large differences would suggest an unlucky split.
comparison_features = [
    "AnnualSalary", "Savings", "RetirementAccountBalance", "EmergencyFund",
    "Age", "SavingsRate", "CreditScore", "MonthlyExpenses",
]

comparison_rows = []
for feature in comparison_features:
    train_median = X_train[feature].median()
    test_median = X_test[feature].median()
    comparison_rows.append({
        "feature": feature,
        "train_median": train_median,
        "test_median": test_median,
        "pct_difference": (test_median / train_median - 1) * 100,
    })

split_comparison = pd.DataFrame(comparison_rows).set_index("feature")
split_comparison.round(3)

The selected variables show no large differences between the training and test samples,
supporting the use of this random split.

---

# 7. Handle Missing Values

### What

Some customers are missing values for certain financial variables. Rather than removing those
customers, a missing numeric value is replaced with the median value learned from the training
data, and a missing category with the most common category in the training data.

### Why

Dropping every incomplete row would discard roughly one customer in six in order to repair well
under one percent of the individual values. Because the observed missingness pattern is
consistent with MCAR, median imputation is a reasonable simple baseline. It is also robust to the
skew in several financial variables: the average balance is pulled upward by a small number of
very wealthy customers, whereas the median reflects a typical customer.

### Decision

`SimpleImputer(strategy="median")` for numeric columns and
`SimpleImputer(strategy="most_frequent")` for categorical columns. Both are created here and
fitted later as steps inside the pipeline, so the replacement values come from the training data
alone.

In [ ]:
train_missing = X_train.isna().sum()
affected_columns = train_missing[train_missing > 0].index.tolist()

missing_rows = []
for column in affected_columns:
    missing_rows.append({
        "column": column,
        "n_missing": int(X_train[column].isna().sum()),
        "pct_missing": X_train[column].isna().mean() * 100,
        "train_median": X_train[column].median(),
        "train_mean": X_train[column].mean(),
    })

missing_summary = pd.DataFrame(missing_rows).set_index("column")

print(f"Numeric columns with gaps:     {len(affected_columns)}")
print(f"Missing values in training set: {X_train.isna().sum().sum():,} "
      f"({X_train.isna().sum().sum() / X_train.size * 100:.2f}% of all values)")
print(f"Training rows with any gap:     {X_train.isna().any(axis=1).sum():,} "
      f"({X_train.isna().any(axis=1).mean() * 100:.1f}%)")
print()
missing_summary.round(2)

In [ ]:
numeric_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

---

# 8. Feature Engineering

### What

Five new columns are calculated from columns the dataset already contains, and the three columns
those calculations make redundant are removed.

### Why

A model does not automatically combine columns. A linear model can only add its inputs together
with weights, so a difference, a product or a ratio has to be supplied as its own column. Each of
the five below is something a financial adviser would look at, which keeps the set small enough to
explain and to debug.

| Feature | Calculation | What it represents |
|---|---|---|
| `YearsUntilRetirement` | `DesiredRetirementAge − Age` | Years the customer's money still has to grow. |
| `CareerStartAge` | `Age − YearsExperience` | Age at which the customer started working. `Age` and `YearsExperience` correlate at +0.94; their difference separates them. |
| `SalaryBasedContribution` | `AnnualSalary × SavingsRate` | The salary-based savings amount implied by `SavingsRate` — a proxy, not a recorded contribution. |
| `RealExpectedReturn` | `ExpectedAnnualReturn − ExpectedInflation` | Investment growth after inflation. Inflation alone barely relates to the target; it matters relative to the return. |
| `DebtToIncomeRatio` | `MortgageBalance ÷ AnnualSalary` | Mortgage size relative to earnings. The same balance is manageable on a high salary and severe on a low one. |

### Decision

`DesiredRetirementAge`, `YearsExperience` and `ExpectedInflation` are dropped once the five are
built: each is an exact component of a new column, so keeping both stores the same information
twice and makes regularised linear coefficients unstable. Nothing is removed merely for being
correlated.

The calculations run after imputation, so a customer missing one input still gets features built
from the rest of their record. Each uses only values from a single row, so this step learns
nothing from the data.

### Customers with no salary

835 customers have `AnnualSalary == 0` — the unemployed group rather than damaged records — and
dividing by zero would make `DebtToIncomeRatio` infinite. The denominator is treated as missing
for them, turning the ratio into an ordinary missing value that a second median imputer fills.
`EmploymentStatus_Unemployed` survives encoding, so the model still knows who they are.

In [ ]:
feature_engineering_step = FunctionTransformer(
    add_engineered_features,
    feature_names_out=engineered_feature_names,
)

# Fills DebtToIncomeRatio for the customers with no salary.
engineered_imputer = SimpleImputer(strategy="median")

print("Engineered:", ", ".join(ENGINEERED_FEATURES))
print("Removed as exact components:", ", ".join(SOURCE_COLUMNS_TO_DROP))

In [ ]:
# Preview on the raw training columns. Inside the pipeline, engineering runs after
# numeric imputation, so the only new missing values it can introduce are undefined
# debt-to-income ratios for customers with zero salary.
engineered_preview = add_engineered_features(X_train[NUMERIC_FEATURES])

preview = engineered_preview[ENGINEERED_FEATURES].describe().T[["min", "50%", "max"]]

correlations = []
for feature in ENGINEERED_FEATURES:
    correlations.append(engineered_preview[feature].corr(y_train))
preview["corr_with_log_target"] = correlations

zero_salary_rows = X_train.AnnualSalary == 0
print(f"Training customers with no salary: {zero_salary_rows.sum():,} "
      f"({zero_salary_rows.mean() * 100:.2f}%)")
print("Their employment status:",
      X_train.loc[zero_salary_rows, "EmploymentStatus"].unique())
print("Infinite values produced:",
      int(np.isinf(engineered_preview[ENGINEERED_FEATURES]).sum().sum()))
print()
preview.round(4)

---

# 9. Encode Categorical Variables

The model requires numerical inputs, so categorical values such as `Education` and
`EmploymentStatus` are converted into binary indicator columns — one column per category, holding
1 if the customer belongs to it and 0 otherwise.

Dropping the first level of each column avoids unnecessary perfect redundancy, since the dropped
category is already implied when all the others are 0. Ignoring unknown categories prevents a
value that never appeared in training from breaking the pipeline in production; such a value is
encoded as all zeros.

In [ ]:
encoding_rows = []
for column in CATEGORICAL_FEATURES:
    levels = sorted(X_train[column].dropna().unique())
    encoding_rows.append({
        "column": column,
        "n_levels": len(levels),
        # Categories are sorted, so drop="first" removes this one.
        "dropped_level": levels[0],
        "columns_created": len(levels) - 1,
    })

encoding_plan = pd.DataFrame(encoding_rows).set_index("column")
print(f"Indicator columns to be created: {encoding_plan.columns_created.sum()}")
print()
encoding_plan

In [ ]:
categorical_encoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore",
    sparse_output=False,
)

---

# 10. Scale Numerical Features

The numeric columns are measured on very different scales — savings rates sit between 0 and 1
while retirement balances run into the millions. Standardising rewrites each column in terms of
how far a value sits from that column's average, measured in standard deviations, so all of them
become comparable.

Ordinary Linear Regression does not mathematically require scaling, but scaling provides a
consistent numerical representation. Scaling is particularly important for regularised linear
models because their penalties depend on coefficient magnitude, and for distance-based methods
such as SVM and KNN. Tree-based models do not depend on feature scale, so they can use the
unscaled version.

Two variants of the pipeline are therefore built: one with scaling and one without.

In [ ]:
scale_overview = pd.DataFrame({
    "min": X_train[NUMERIC_FEATURES].min(),
    "median": X_train[NUMERIC_FEATURES].median(),
    "max": X_train[NUMERIC_FEATURES].max(),
    "std": X_train[NUMERIC_FEATURES].std(),
}).sort_values("std")

print(f"Smallest standard deviation: {scale_overview['std'].min():,.4f} "
      f"({scale_overview.index[0]})")
print(f"Largest standard deviation:  {scale_overview['std'].max():,.0f} "
      f"({scale_overview.index[-1]})")
print()
scale_overview.round(4)

In [ ]:
feature_scaler = StandardScaler()

---

# 11. Build the Preprocessing Pipeline

The steps defined above are now assembled into one object. Numeric and categorical columns follow
separate routes and are joined back together at the end.

Keeping everything in a single pipeline means the steps always run in the right order, the
training and prediction paths are identical, and cross-validation refits the whole sequence
correctly on each fold. `remainder="drop"` is stated explicitly so a column that is not assigned
to a branch is discarded rather than passed through unnoticed.

In [ ]:
# Numeric route: fill gaps, build features, fill the undefined ratio, scale.
numeric_pipeline = Pipeline(steps=[
    ("impute", numeric_imputer),
    ("engineer", feature_engineering_step),
    ("impute_engineered", engineered_imputer),
    ("scale", feature_scaler),
])

# Categorical route: fill gaps, convert to indicator columns.
categorical_pipeline = Pipeline(steps=[
    ("impute", categorical_imputer),
    ("encode", categorical_encoder),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,   # keep "Age", not "numeric__Age"
)

# Return DataFrames so column names survive the transformation.
preprocessor.set_output(transform="pandas")

preprocessor

In [ ]:
# The same pipeline with the scaling step switched off, for tree-based models.
preprocessor_unscaled = clone(preprocessor)
preprocessor_unscaled.set_params(numeric__scale="passthrough")
preprocessor_unscaled.set_output(transform="pandas")

print("scaled variant   ->", preprocessor.get_params()["numeric__scale"])
print("unscaled variant ->", preprocessor_unscaled.get_params()["numeric__scale"])

# How wide the output should be, derived from the configuration above. The
# verification section compares this with the matrix that is actually produced.
n_numeric_kept = len(NUMERIC_FEATURES) - len(SOURCE_COLUMNS_TO_DROP)
n_engineered = len(ENGINEERED_FEATURES)
n_numeric_total = n_numeric_kept + n_engineered
n_dummies = int(encoding_plan.columns_created.sum())
n_expected_features = n_numeric_total + n_dummies

print(f"\nExpected: {n_numeric_total} numeric + {n_dummies} indicator "
      f"= {n_expected_features} features")

In [ ]:
# --- Figure: the two routes through the pipeline --------------------------
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 100)
ax.set_ylim(0, 52)
ax.axis("off")


def draw_box(x, y, width, height, label, colour):
    ax.add_patch(plt.Rectangle((x, y), width, height, facecolor=colour,
                               edgecolor="white", linewidth=1.6, zorder=2))
    ax.text(x + width / 2, y + height / 2, label, ha="center", va="center",
            fontsize=9.5, color="white", fontweight="bold", zorder=3)


def draw_arrow(x_start, y, x_end):
    ax.annotate("", xy=(x_end, y), xytext=(x_start, y),
                arrowprops={"arrowstyle": "->", "color": SLATE, "linewidth": 1.6})


draw_box(1, 21, 13, 8, f"X_train\n{X_train.shape[0]:,} x {X_train.shape[1]}", NAVY)
draw_arrow(14.4, 25, 16.2)

ax.text(18, 45.5, f"NUMERIC  ({len(NUMERIC_FEATURES)} columns)", fontsize=10,
        color=NAVY, fontweight="bold")
numeric_stages = ["Fill gaps\n(median)", "Build 5 features\nremove 3 sources",
                  "Fill undefined\nratio", "Standardise"]
for index, stage in enumerate(numeric_stages):
    x = 18 + index * 16
    draw_box(x, 34, 13.5, 9, stage, TEAL)
    if index < len(numeric_stages) - 1:
        draw_arrow(x + 13.9, 38.5, x + 15.6)

ax.text(18, 15.5, f"CATEGORICAL  ({len(CATEGORICAL_FEATURES)} columns)",
        fontsize=10, color=NAVY, fontweight="bold")
categorical_stages = ["Fill gaps\n(most frequent)", "Indicator columns\ndrop first"]
for index, stage in enumerate(categorical_stages):
    x = 18 + index * 16
    draw_box(x, 4, 13.5, 9, stage, AMBER)
    if index < len(categorical_stages) - 1:
        draw_arrow(x + 13.9, 8.5, x + 15.6)

# Connectors splitting the input into two routes and merging them again.
ax.plot([16.2, 16.2], [8.5, 38.5], color=SLATE, lw=1.6, zorder=1)
ax.plot([16.2, 18], [38.5, 38.5], color=SLATE, lw=1.6, zorder=1)
ax.plot([16.2, 18], [8.5, 8.5], color=SLATE, lw=1.6, zorder=1)
ax.plot([79.5, 84], [38.5, 38.5], color=SLATE, lw=1.6, zorder=1)
ax.plot([47.5, 84], [8.5, 8.5], color=SLATE, lw=1.6, zorder=1)
ax.plot([84, 84], [8.5, 38.5], color=SLATE, lw=1.6, zorder=1)
draw_arrow(84, 25, 86)

draw_box(86, 21, 13, 8,
         f"processed\n{X_train.shape[0]:,} x {n_expected_features}", NAVY)

for x, label in [(24.8, len(NUMERIC_FEATURES)), (40.8, n_numeric_total),
                 (56.8, n_numeric_total), (72.8, n_numeric_total)]:
    ax.text(x, 31.5, f"{label} cols", ha="center", fontsize=8.5, color=SLATE)
for x, label in [(24.8, len(CATEGORICAL_FEATURES)), (40.8, n_dummies)]:
    ax.text(x, 1.2, f"{label} cols", ha="center", fontsize=8.5, color=SLATE)

ax.text(50, 21.5, "fitted on X_train    |    applied to X_train and X_test",
        ha="center", fontsize=11, color=CORAL, fontweight="bold")
ax.text(50, 18.2, "medians, scaling statistics and category lists all come "
        "from the training data",
        ha="center", fontsize=9, color=SLATE, style="italic")

ax.set_title("Preprocessing pipeline", fontsize=14, fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_preprocessing_pipeline.png", dpi=300,
            bbox_inches="tight")
plt.show()

---

# 12. Transform the Data

`fit_transform` learns the medians, scaling statistics and category lists from the training set
and then applies them. `transform` applies those same values to the test set without recalculating
anything, which is what keeps the test set a fair estimate of future performance.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_train_unscaled = preprocessor_unscaled.fit_transform(X_train)
X_test_unscaled = preprocessor_unscaled.transform(X_test)

print(f"X_train_processed: {X_train_processed.shape}")
print(f"X_test_processed:  {X_test_processed.shape}")
print(f"X_train_unscaled:  {X_train_unscaled.shape}")
print(f"X_test_unscaled:   {X_test_unscaled.shape}")

---

# 13. Verify the Final Dataset

Feature counts are read from the transformed matrix rather than typed by hand, and each property
is asserted so that a future change stops the notebook instead of producing a quietly broken
dataset.

In [ ]:
# Counts are read from the transformed matrix, not typed in.
feature_names = list(X_train_processed.columns)
n_indicators = len(feature_names) - n_numeric_total

print(f"Raw predictor columns:    {X.shape[1]}")
print(f"  original numeric kept:  {n_numeric_kept}")
print(f"  engineered:             {n_engineered}")
print(f"  categorical indicators: {n_indicators}")
print(f"Final model features:     {len(feature_names)}")
print()
print("Numeric features:")
print("   " + ", ".join(feature_names[:n_numeric_total]))
print("Indicator features:")
print("   " + ", ".join(feature_names[n_numeric_total:]))

In [ ]:
# The medians stored by the fitted imputer should match the training medians.
fitted_imputer = preprocessor.named_transformers_["numeric"].named_steps["impute"]
learned_medians = pd.Series(fitted_imputer.statistics_, index=NUMERIC_FEATURES)
imputer_from_train = np.allclose(learned_medians[affected_columns],
                                 X_train[affected_columns].median())

leakage_columns = []
for name in feature_names:
    if name in KPI_COLUMNS + IDENTIFIER_COLUMNS + [TARGET]:
        leakage_columns.append(name)

remaining_sources = []
for column in SOURCE_COLUMNS_TO_DROP:
    if column in feature_names:
        remaining_sources.append(column)

# Exact duplicate records were removed before the train/test split.
assert df.duplicated().sum() == 0, "Exact duplicate records remain in the modeling data."

checks = [
    ("No missing values in train", int(X_train_processed.isna().sum().sum()) == 0),
    ("No missing values in test", int(X_test_processed.isna().sum().sum()) == 0),
    ("No infinite values", bool(np.isfinite(X_train_processed.to_numpy()).all()
                                and np.isfinite(X_test_processed.to_numpy()).all())),
    ("No identifier or KPI columns", len(leakage_columns) == 0),
    ("No exact source columns retained", len(remaining_sources) == 0),
    ("Train and test columns identical",
     list(X_train_processed.columns) == list(X_test_processed.columns)),
    ("Feature count matches the pipeline definition",
     len(feature_names) == n_expected_features),
    ("Imputer values learned from training data only", bool(imputer_from_train)),
]

results = pd.DataFrame(checks, columns=["check", "passed"]).set_index("check")
print(results.to_string())

assert results.passed.all(), "One or more preprocessing checks failed."
print("\nAll checks passed.")

---

# 14. Save Preprocessing Artifacts

The fitted pipelines, the feature names, the raw splits and the two versions of the target are
written to `artifacts/`. The business figures are not saved here: they belong after the
prediction, not before it.

The saved preprocessing pipeline is then reloaded in a separate Python process to confirm that it
does not depend on variables stored only in this notebook.

In [ ]:
joblib.dump(preprocessor, ARTIFACTS_DIR / "preprocessor.joblib")
joblib.dump(preprocessor_unscaled, ARTIFACTS_DIR / "preprocessor_unscaled.joblib")

pd.Series(feature_names, name="feature").to_csv(
    ARTIFACTS_DIR / "feature_names.csv", index=False)

# Raw splits, so Notebook 3 can rebuild any matrix through the pipeline.
X_train.to_csv(ARTIFACTS_DIR / "X_train.csv.gz")
X_test.to_csv(ARTIFACTS_DIR / "X_test.csv.gz")

pd.DataFrame({"log_target": y_train, "target_dollars": y_train_dollars}).to_csv(
    ARTIFACTS_DIR / "y_train.csv")
pd.DataFrame({"log_target": y_test, "target_dollars": y_test_dollars}).to_csv(
    ARTIFACTS_DIR / "y_test.csv")

for artifact_path in sorted(ARTIFACTS_DIR.iterdir()):
    size_mb = artifact_path.stat().st_size / 1024**2
    print(f"   {artifact_path.name:<30} {size_mb:>6.2f} MB")

In [ ]:
# Load the pipeline in a separate process and re-transform the test set there.
reload_script = """
import joblib
import pandas as pd

preprocessor = joblib.load("artifacts/preprocessor.joblib")
X_test = pd.read_csv("artifacts/X_test.csv.gz", index_col=0)
transformed = preprocessor.transform(X_test)

print(transformed.shape[0])
print(transformed.shape[1])
print(",".join(transformed.columns))
"""

result = subprocess.run(
    [sys.executable, "-c", reload_script],
    cwd=PROJECT_ROOT, capture_output=True, text=True,
)

if result.returncode != 0:
    raise RuntimeError(f"Reload failed:\n{result.stderr}")

reloaded_rows, reloaded_columns, reloaded_names = result.stdout.strip().split("\n")

print(f"Rows reproduced:          {int(reloaded_rows) == X_test_processed.shape[0]}")
print(f"Columns reproduced:       {int(reloaded_columns) == X_test_processed.shape[1]}")
print(f"Feature names reproduced: {reloaded_names.split(',') == feature_names}")

assert int(reloaded_rows) == X_test_processed.shape[0]
assert int(reloaded_columns) == X_test_processed.shape[1]
assert reloaded_names.split(",") == feature_names
print("\nThe saved pipeline reproduces the test matrix in a separate process.")

---

# 15. Summary

**Removed.** `CustomerID`, which identifies a customer without describing one, and `Funding_Gap`,
`Readiness_Score` and `RetirementReady`, which are calculated from the target and so cannot serve
as inputs. 150 duplicate records were removed before the split. `DesiredRetirementAge`,
`YearsExperience` and `ExpectedInflation` were removed after feature engineering because each is
an exact component of a new feature.

**Engineered.** `YearsUntilRetirement`, `CareerStartAge`, `SalaryBasedContribution`,
`RealExpectedReturn` and `DebtToIncomeRatio`. Customers with no salary have no defined
debt-to-income ratio, so that value is treated as missing and imputed rather than left as
infinity.

**Missing values.** Numeric gaps filled with the training median, categorical gaps with the most
common training category.

**Categorical variables.** Converted to binary indicator columns, dropping one level per column to
avoid redundancy and ignoring unseen categories so new values cannot break the pipeline.

**Scaling.** A standardised version for regularised linear and distance-based models, and an
unscaled version for tree-based models.

**Contamination control.** The split happens before anything is fitted, and every value the
pipeline learns comes from the training data. The test set is only ever transformed.

**Artifacts saved.** `preprocessor.joblib`, `preprocessor_unscaled.joblib`, `feature_names.csv`,
the raw splits and both versions of the target. The feature-engineering code lives in
`src/feature_engineering.py`, so the pipeline reloads cleanly outside this notebook.

**Assumption to revisit.** `Retirement_Fund_Goal` is treated as known before prediction; Notebook 3
tests that with an ablation run.

The data is now prepared for baseline model training and comparison in Notebook 3.